### c) Compute sample report times

In [10]:
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize_scalar

def sample_nhppp(l, T):
    """Simulate one NHPPP realization on [0,T] using thinning"""

    # Finding max lambda
    res = minimize_scalar(lambda x: -l(x), bounds=(0, T), method='bounded')
    lmax = -res.fun

    # Finding all arrival times
    t = 0
    arrivals = []

    while True:

        # exponential interarrival
        t += np.random.exponential(1 / lmax)

        # Ending when enough have been found
        if t > T:
            break

        # accept with probability related to l(t)
        if np.random.uniform() < l(t) / lmax:
            arrivals.append(t)

    return np.array(arrivals)


# Givens
l = lambda t: 0.5 * (1 + (t/30)**2)
T = 120

# simulation numbers
n_sims = 500
all_samples = []

for _ in range(n_sims):
    all_samples.extend(sample_nhppp(l, T))

all_samples = np.array(all_samples)

# bins = days
bins = np.arange(0, T + 1, 1)

hist, edges = np.histogram(all_samples, bins=bins)

# average per realization (expected reports per day)
hist_avg = hist / n_sims

# midpoints of bins
t_mid = 0.5 * (edges[:-1] + edges[1:])

# theoretical lambda(t)
lambda_vals = l(t_mid)

fig = go.Figure()

# empirical histogram (expected counts per day)
fig.add_trace(go.Bar(
    x=t_mid,
    y=hist_avg,
    name="Simulated avg counts/day",
    opacity=0.6
))

# theoretical rate lambda(t)
fig.add_trace(go.Scatter(
    x=t_mid,
    y=lambda_vals,
    mode='lines',
    name="lambda(t)",
    line=dict(width=3)
))

fig.update_layout(
    title="NHPPP Simulated vs Theoretical Rate",
    xaxis_title="Time (days)",
    yaxis_title="Reports per day",
    bargap=0
)

fig.show()